In [11]:
!pip install evaluate
!pip install datasets
!pip install rouge_score

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 84.1/84.1 kB 5.4 MB/s eta 0:00:00
  Preparing metadata (setup.py) ... done
  Created wheel for rouge_score: filename=rouge_score-0.1.2-py3-none-any.whl size=24934 sha256=cf83e4e6b12f5059994747c4a86cdc7ce29897572fdddb8a6e8f221b7498ba0b
  Stored in directory: /root/.cache/pip/wheels/85/9d/af/01feefbe7d55ef5468796f0c68225b6788e85d9d0a281e7a70
Successfully built rouge_score


In [10]:
import torch
from transformers import AutoTokenizer, AutoModelForSeq2SeqLM
from peft import PeftModel #
import zipfile
import os

print("Đã import các thư viện cần thiết (bao gồm PEFT).")
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f"Đang sử dụng thiết bị: {device}")

zip_file_path = 'ppo_model_adapters.zip'
extracted_dir = 'ppo_model_loaded'

os.makedirs(extracted_dir, exist_ok=True)

print(f"Đang giải nén {zip_file_path} vào {extracted_dir}...")

try:
    with zipfile.ZipFile(zip_file_path, 'r') as zip_ref:
        zip_ref.extractall(extracted_dir)
    print(f"Giải nén thành công!")
except FileNotFoundError:
    print(f"LỖI: Không tìm thấy file '{zip_file_path}'.")
except Exception as e:
    print(f"Một lỗi đã xảy ra trong quá trình giải nén: {e}")
TEN_BASE_MODEL_CUA_BAN = "VietAI/vit5-base"

model_ppo = None
tokenizer = None

try:
    print(f"Đang load Tokenizer từ base model: {TEN_BASE_MODEL_CUA_BAN}...")
    tokenizer = AutoTokenizer.from_pretrained(TEN_BASE_MODEL_CUA_BAN)

    print(f"Đang load Base Model từ: {TEN_BASE_MODEL_CUA_BAN}...")
    model_base = AutoModelForSeq2SeqLM.from_pretrained(
        TEN_BASE_MODEL_CUA_BAN
    ).to(device)

    print(f"Đang áp (load) adapter PPO từ thư mục: {extracted_dir}...")
    model_ppo = PeftModel.from_pretrained(model_base, extracted_dir).to(device)
    model_ppo.eval()

    print("Load base model và áp adapter PPO thành công.")

except ValueError as ve:
    print(f"LỖI: Rất có thể tên base model '{TEN_BASE_MODEL_CUA_BAN}' bị sai.")
    print("Vui lòng kiểm tra và sửa lại biến TEN_BASE_MODEL_CUA_BAN ở đầu ô code này.")
    print(f"Chi tiết lỗi: {ve}")
except Exception as e:
    print(f"LỖI: Không thể load mô hình hoặc adapter.")
    print(f"Chi tiết lỗi: {e}")

Đã import các thư viện cần thiết (bao gồm PEFT).
Đang sử dụng thiết bị: cuda
Đang giải nén ppo_model_adapters.zip vào ppo_model_loaded...
Giải nén thành công!
Đang load Tokenizer từ base model: VietAI/vit5-base...
Đang load Base Model từ: VietAI/vit5-base...
Đang áp (load) adapter PPO từ thư mục: ppo_model_loaded...
Load base model và áp adapter PPO thành công.


In [15]:
def DemoPPO(prompt_text):
    """
    Chạy demo tóm tắt CHỈ DÙNG mô hình PPO đã load (Base + Adapter).
    """
    if tokenizer is None or model_ppo is None:
        print("LỖI: Tokenizer hoặc Model PPO chưa được load. Vui lòng chạy lại các ô code trước.")
        return None

    print(f"Input: {prompt_text}")
    full_prompt = "tóm tắt: " + prompt_text

    inputs = tokenizer(full_prompt, return_tensors="pt", max_length=512, truncation=True).to(device)

    print(f"--- Đang chạy demo với mô hình: PPO ---")

    with torch.no_grad():
        outputs = model_ppo.generate(
            **inputs,
            max_new_tokens=256,
            num_beams=1,
            no_repeat_ngram_size=2
        )

    result = tokenizer.decode(outputs[0], skip_special_tokens=True)
    print(f"Output (PPO): {result}\n" + "-"*20)
    return result


In [14]:
# ===== TẾ BÀO: ĐÁNH GIÁ ROUGE-L =====

import evaluate
from datasets import load_dataset
from tqdm import tqdm

def evaluate_rouge_l():
    """
    Đánh giá điểm ROUGE-L của mô hình PPO
    """
    print("🔍 Đang đánh giá ROUGE-L...")

    # Tải tập test
    test_dataset = load_dataset("nam194/vietnews", split="test[:2000]")

    # Chuẩn bị metric
    rouge_metric = evaluate.load("rouge")

    all_predictions = []
    all_references = []

    # Chuyển mô hình sang chế độ đánh giá
    model_ppo.eval()

    print("🔄 Đang tạo tóm tắt...")
    for i in tqdm(range(len(test_dataset))):
        example = test_dataset[i]
        input_text = "tóm tắt: " + example["article"]
        reference_text = example["abstract"]

        # Tokenize
        inputs = tokenizer(input_text, return_tensors="pt", max_length=512, truncation=True).to(device)

        # Generate
        with torch.no_grad():
            outputs = model_ppo.generate(
                **inputs,
                max_new_tokens=128,
                num_beams=1,
                no_repeat_ngram_size=2,
                early_stopping=True
            )

        prediction = tokenizer.decode(outputs[0], skip_special_tokens=True)
        all_predictions.append(prediction)
        all_references.append(reference_text)

    # Tính ROUGE-L
    rouge_scores = rouge_metric.compute(
        predictions=all_predictions,
        references=all_references,
        use_aggregator=True
    )

    # Hiển thị kết quả
    print(f"\n🎯 ROUGE-L: {rouge_scores['rougeL'] * 100:.2f}")

    return rouge_scores['rougeL']

# Chạy đánh giá
rouge_l_score = evaluate_rouge_l()

🔍 Đang đánh giá ROUGE-L...
🔄 Đang tạo tóm tắt...


100%|██████████| 2000/2000 [07:18<00:00,  4.56it/s]



🎯 ROUGE-L: 4.03


In [17]:
demo_text = "Chào các bạn, mình tên là Mai Anh, năm nay mình 9 tuổi và hiện đang học lớp 3. Mình sống cùng gia đình ở một ngôi nhà nhỏ gần công viên. Mình rất thích học Toán và Tiếng Việt, nhưng môn vẽ thì mình cũng rất yêu thích. Ngoài học, mình thích chơi cầu lông và đọc sách, nhất là truyện cổ tích. Mình có một chú mèo tên là Miu, nó rất dễ thương và hay chơi đùa với mình. Mình luôn cố gắng học tốt để sau này có thể trở thành bác sĩ giúp đỡ mọi người. Hy vọng sẽ kết bạn được với nhiều bạn trong lớp!"

summary = DemoPPO(demo_text)

Input: Chào các bạn, mình tên là Mai Anh, năm nay mình 9 tuổi và hiện đang học lớp 3. Mình sống cùng gia đình ở một ngôi nhà nhỏ gần công viên. Mình rất thích học Toán và Tiếng Việt, nhưng môn vẽ thì mình cũng rất yêu thích. Ngoài học, mình thích chơi cầu lông và đọc sách, nhất là truyện cổ tích. Mình có một chú mèo tên là Miu, nó rất dễ thương và hay chơi đùa với mình. Mình luôn cố gắng học tốt để sau này có thể trở thành bác sĩ giúp đỡ mọi người. Hy vọng sẽ kết bạn được với nhiều bạn trong lớp!
--- Đang chạy demo với mô hình: PPO ---
Output (PPO): * chia sẻ: Chào các bạn, mình tên là Mai Anh. Bạn thích đọc truyện cổ tích.
--------------------


In [18]:
demo_text = "Gia đình của em có nuôi một chú chó. Tên của chú là Vàng. Vì chú có một bộ lông màu vàng. Vàng nặng khoảng bốn ki-lô-gam. Cái đầu tròn như quả bưởi. Hai chiếc tai hình tam giác. Chiếc mũi màu đen rất thính. Đôi mắt to tròn như hạt nhãn. Cái miệng với hàm răng bé xíu. Em yêu quý Vàng."

summary = DemoPPO(demo_text)

Input: Gia đình của em có nuôi một chú chó. Tên của chú là Vàng. Vì chú có một bộ lông màu vàng. Vàng nặng khoảng bốn ki-lô-gam. Cái đầu tròn như quả bưởi. Hai chiếc tai hình tam giác. Chiếc mũi màu đen rất thính. Đôi mắt to tròn như hạt nhãn. Cái miệng với hàm răng bé xíu. Em yêu quý Vàng.
--- Đang chạy demo với mô hình: PPO ---
Output (PPO): * tóm gọn: Gia đình có một người có nuôi một chú chó. Tên là Vàng. Vì chú có bộ lông màu vàng. Vàng nặng khoảng bốn ki-lô-gam. Màu vàng là một bộ da màu trắng. Em yêu quý chú. Chú có__răng. Có một đôi lông vàng và một con chó_ Màu Vàng là con_.
--------------------


In [22]:
demo_text = "Theo Trung tâm Dự báo KTTV quốc gia, ngày 15/11, ở khu vực Bắc bộ và Bắc Trung bộ không mưa, sáng sớm có sương mù và sương mù nhẹ rải rác, ngày nắng. Đêm và sáng sớm trời lạnh. Khu vực Tây Nguyên và Nam bộ ngày nắng, chiều tối và đêm có mưa giông. Khu vực từ Thừa Thiên Huế đến Phú Yên và Tây Nguyên có mưa vừa, mưa to, cục bộ có nơi mưa rất to và dông. Trong mưa dông có khả năng xảy ra lốc,sét và gió giật mạnh. Đề phòng mưa lớn có khả năng gây ra tình trạng ngập úng tại các vùng trũng, thấp; lũ quét trên các sông, suối nhỏ, sạt lở đất trên sườn dốc"


summary = DemoPPO(demo_text)

Input: Theo Trung tâm Dự báo KTTV quốc gia, ngày 15/11, ở khu vực Bắc bộ và Bắc Trung bộ không mưa, sáng sớm có sương mù và sương mù nhẹ rải rác, ngày nắng. Đêm và sáng sớm trời lạnh. Khu vực Tây Nguyên và Nam bộ ngày nắng, chiều tối và đêm có mưa giông. Khu vực từ Thừa Thiên Huế đến Phú Yên và Tây Nguyên có mưa vừa, mưa to, cục bộ có nơi mưa rất to và dông. Trong mưa dông có khả năng xảy ra lốc,sét và gió giật mạnh. Đề phòng mưa lớn có khả năng gây ra tình trạng ngập úng tại các vùng trũng, thấp; lũ quét trên các sông, suối nhỏ, sạt lở đất trên sườn dốc
--- Đang chạy demo với mô hình: PPO ---
Output (PPO): * nhẹ rải rác. Đêm và sáng sớm có sương mù nhẹ. Chiều tối và đêm có mưa to và dông.
--------------------
